# 00 Colab Runbook - AI-Powered Vehicle Damage Assessment Pipeline

This notebook prepares the Colab runtime, syncs the latest GitHub project code to Google Drive, and runs the vehicle damage assessment pipeline through `!python -m ...` CLI commands.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Sync Latest Project Code From GitHub

Run this when you want the Drive project folder to match the latest GitHub repository code. This keeps datasets, checkpoints, and generated runs in Drive.


In [ ]:
%cd /content
!rm -rf ai-powered-vehicle-damage-assessment-pipeline
!git clone https://github.com/lsdlBlueDragon/ai-powered-vehicle-damage-assessment-pipeline.git

!mkdir -p /content/drive/MyDrive/CarDD_YOLO11
!cp -r ai-powered-vehicle-damage-assessment-pipeline/src /content/drive/MyDrive/CarDD_YOLO11/
!cp -r ai-powered-vehicle-damage-assessment-pipeline/tests /content/drive/MyDrive/CarDD_YOLO11/
!cp -r ai-powered-vehicle-damage-assessment-pipeline/docs /content/drive/MyDrive/CarDD_YOLO11/
!cp -r ai-powered-vehicle-damage-assessment-pipeline/notebooks /content/drive/MyDrive/CarDD_YOLO11/
!cp -r ai-powered-vehicle-damage-assessment-pipeline/configs /content/drive/MyDrive/CarDD_YOLO11/
!cp ai-powered-vehicle-damage-assessment-pipeline/README.md /content/drive/MyDrive/CarDD_YOLO11/
!cp ai-powered-vehicle-damage-assessment-pipeline/pyproject.toml /content/drive/MyDrive/CarDD_YOLO11/
!cp ai-powered-vehicle-damage-assessment-pipeline/requirements-colab.txt /content/drive/MyDrive/CarDD_YOLO11/


## 3. Install Dependencies and Editable Package


In [ ]:
%cd /content/drive/MyDrive/CarDD_YOLO11
!pip install -q -r requirements-colab.txt
!pip install -q -e .
!python -m pytest tests -q -p no:cacheprovider


## 4. Quick Validation With Existing Drive Results

Use this path first if YOLO training, demo inference, and Qwen report artifacts already exist in Drive.


In [ ]:
!python -m vehicle_damage_pipeline.report.build_context --drive-root /content/drive/MyDrive/CarDD_YOLO11 --output /content/drive/MyDrive/CarDD_YOLO11/reports/qwen7b_report_context.json
!python -m vehicle_damage_pipeline.report.generate --context /content/drive/MyDrive/CarDD_YOLO11/reports/qwen7b_report_context.json --output /content/drive/MyDrive/CarDD_YOLO11/reports/qwen7b_final_report.md --language Chinese
!python -m vehicle_damage_pipeline.eval.run_llm_eval --context /content/drive/MyDrive/CarDD_YOLO11/reports/qwen7b_report_context.json --report /content/drive/MyDrive/CarDD_YOLO11/reports/qwen7b_final_report.md --knowledge-root /content/drive/MyDrive/CarDD_YOLO11 --output-json /content/drive/MyDrive/CarDD_YOLO11/reports/llm_eval_summary.json --output-markdown /content/drive/MyDrive/CarDD_YOLO11/reports/llm_eval_summary.md


## 5. Run Demo Prediction


In [ ]:
from pathlib import Path
weights = '/content/drive/MyDrive/CarDD_YOLO11/runs/train/yolo11n_seg/weights/best.pt'
source = '/content/drive/MyDrive/CarDD_YOLO11/demo_images'
output = '/content/drive/MyDrive/CarDD_YOLO11/runs/predict/demo'
if not any(Path(source).glob('*')):
    source = '/content/drive/MyDrive/CarDD_YOLO11/data_yolo/images/test'

!python -m vehicle_damage_pipeline.vision.predict --weights "{weights}" --source "{source}" --output "{output}" --imgsz 1024 --conf 0.25 --iou 0.7


## 6. Full YOLO Training Path

Run this only when you want to prepare data and train/evaluate/export YOLO again. It can take several hours on Colab L4.


In [ ]:
!python -m vehicle_damage_pipeline.data.prepare_cardd --drive-root /content/drive/MyDrive/CarDD_YOLO11
!python -m vehicle_damage_pipeline.vision.train_yolo --drive-root /content/drive/MyDrive/CarDD_YOLO11 --model yolo11n-seg.pt --run-name yolo11n_seg --epochs 100 --imgsz 1024 --batch 7


## 7. Optional Qwen QLoRA Report Adapter Rerun

Run this only if you want to rebuild the LoRA adapter. The training command uses `packing=False` to avoid unsupported attention packing warnings.


In [ ]:
!python -m vehicle_damage_pipeline.llm.finetune_report_lora --drive-root /content/drive/MyDrive/CarDD_YOLO11 --build-dataset-only --train-examples 1200 --eval-examples 120
!python -m vehicle_damage_pipeline.llm.finetune_report_lora --drive-root /content/drive/MyDrive/CarDD_YOLO11 --train-examples 1200 --eval-examples 120 --epochs 1 --max-seq-length 1024


## 8. Optional Gradio Demo

Run after installing dependencies and confirming `best.pt` exists.


In [ ]:
!python -m vehicle_damage_pipeline.service.gradio_app --weights /content/drive/MyDrive/CarDD_YOLO11/runs/train/yolo11n_seg/weights/best.pt --output-dir /content/drive/MyDrive/CarDD_YOLO11/runs/gradio_predict


## 9. Inspect Final Report and Evaluation


In [ ]:
from pathlib import Path
for path in [
    Path('/content/drive/MyDrive/CarDD_YOLO11/reports/qwen7b_final_report.md'),
    Path('/content/drive/MyDrive/CarDD_YOLO11/reports/llm_eval_summary.md'),
    Path('/content/drive/MyDrive/CarDD_YOLO11/runs/predict/demo/prediction_summary.json'),
]:
    print('\n---', path, '---')
    if path.exists():
        print(path.read_text(encoding='utf-8')[:3000])
    else:
        print('missing')
